# Mastra + MongoDB Atlas: 4. Agents, Tools & Persistent Memory

The first three notebooks built retrieval. You called it, you read the results, you decided
what to do next. This notebook hands that decision to the model.

Two things change once retrieval sits behind an `Agent`. The model decides *whether* to
search and *what to search for*, which means you no longer control the query, only the
tool's contract. And a conversation now has state worth keeping, because "which of those is
older?" is only answerable if something remembers what "those" referred to. Both halves live
in MongoDB: retrieval through `MongoDBVector`, conversation through `MongoDBStore`.

## What you'll learn
1. Why an agent without tools is confidently unreliable on your data
2. `createTool`: a Zod-validated contract, and what `execute` actually receives
3. Giving the agent two tools and watching which one it picks
4. `Memory` + `MongoDBStore`: threads, resources, and a follow-up that needs context
5. Proving persistence: a brand-new client recalls a conversation it never had

## Prerequisites
- `sample_mflix` loaded, and the setup from notebook 01 (`.env`, `npm install`)
- Claude access: either `ANTHROPIC_API_KEY`, or a gateway via `ANTHROPIC_BASE_URL`

Notebooks 01 to 03 are the natural reading order -- the tools below are the retrievers
those notebooks build -- but the setup cell creates both Atlas indexes if they are absent,
so this notebook runs on its own too.

In [ ]:
import { Agent } from '@mastra/core/agent';
import { createTool } from '@mastra/core/tools';
import { Memory } from '@mastra/memory';
import { MongoDBStore, MongoDBVector } from '@mastra/mongodb';
import { z } from 'zod';

import { loadNotebookConfig } from './src/config.js';
import {
  APP_NAME,
  MFLIX_FULLTEXT_INDEX,
  MFLIX_VECTOR_INDEX,
  buildAnthropicModel,
  connect,
  createAtlasEmbedder,
  ensureFullTextSearchIndex,
  ensureVectorSearchIndex,
} from './src/utils.js';

const cfg = loadNotebookConfig();
console.log('Configuration:', cfg.summary);

const vector = new MongoDBVector({
  id: 'mflix',
  uri: cfg.mongodbUri,
  dbName: cfg.dbName,
  options: { appName: APP_NAME },
  embeddingFieldPath: 'plot_embedding_voyage_3_large',
});
await vector.connect();

// The agent's two retrieval tools need both Atlas indexes. Created if absent, so this
// notebook stands on its own; no-ops if notebooks 01 and 03 already ran.
const { client: indexClient } = await connect(cfg.mongodbUri);
const indexDb = indexClient.db(cfg.dbName);
const movies = indexDb.collection('embedded_movies');

// Passing the database turns on the index-budget preflight: at the cluster's search
// index limit these stop and list what is using the slots, removing nothing.
await ensureVectorSearchIndex(
  movies,
  {
    indexName: MFLIX_VECTOR_INDEX,
    path: 'plot_embedding_voyage_3_large',
    dimensions: cfg.embeddingDimensions,
    filterPaths: ['genres', 'year'],
  },
  { db: indexDb },
);
const TEXT_INDEX = await ensureFullTextSearchIndex(
  movies,
  { indexName: MFLIX_FULLTEXT_INDEX, fields: ['plot'] },
  { db: indexDb },
);
await indexClient.close();

const INDEX = 'mflix_plots';
await vector.createIndex({
  indexName: INDEX,
  dimension: cfg.embeddingDimensions,
  metric: 'cosine',
  collectionName: 'embedded_movies',
  searchIndexName: MFLIX_VECTOR_INDEX,
  filterFields: ['genres', 'year'],
});

const embedder = createAtlasEmbedder({
  apiKey: cfg.atlasApiKey,
  model: cfg.embeddingModel,
  dimensions: cfg.embeddingDimensions,
});

const model = buildAnthropicModel({
  apiKey: cfg.anthropicApiKey,
  model: cfg.anthropicModel,
  baseUrl: cfg.anthropicBaseUrl,
});

console.log('Ready. Model:', cfg.anthropicModel);

Configuration: db=sample_mflix · embeddings=voyage-3-large (2048-d) · llm=claude-haiku-4-5 via gateway


  movies_vector_index already exists.


  movies_fulltext_index already exists.


Ready. Model: claude-haiku-4-5


## 1. An agent with no tools

Start with what an agent does when it has no way to reach your data.

This one has a model and instructions and nothing else. The instructions claim access to a
collection, which is a lie: there is no `tools` key, so there is no collection.

In [ ]:
const unarmed = new Agent({
  id: 'movie-critic-untooled',
  name: 'movie-critic-untooled',
  instructions: 'You are a film expert. Recommend films from the collection you have access to.',
  model: model,
});

const guessed = await unarmed.generate(
  'Name three films in the collection about a submarine crew, with their release years.',
);
console.log(guessed.text);

I don't have access to a specific film collection or database. I can't tell you which films are in a particular collection you're referring to.

However, I can recommend three well-known films about submarine crews:

1. **Das Boot** (1981) - A German war film about a U-boat crew during World War II
2. **The Hunt for Red October** (1990) - About a Soviet submarine captain and his crew during the Cold War
3. **Crimson Tide** (1995) - Featuring a U.S. nuclear submarine crew with tensions between its captain and executive officer

If you have a specific collection in mind, could you provide more details about it? That would help me give you accurate recommendations from that particular source.


Better than it might have been. Claude noticed the contradiction, said plainly that it has no
access to a specific collection, and *then* offered submarine films from general knowledge,
labelled as such. That honesty is a property of the model, not of your architecture.

Look at what it recommended, though: *Das Boot*, *The Hunt for Red October*, *Crimson Tide*,
the canonical submarine films, which is what you get from training data. Two of the three do
happen to be in the searchable collection, and that coincidence is the trap: the answer looks
verified when nothing verified it. Ask about anything less famous and the overlap disappears.
Note too that it hedged here; a weaker model, or the same model under instructions that insist
harder, drops the hedge and keeps the list.

That is the shape of the problem. Instructions cannot grant access. An agent told it has a
database, without a tool that reaches one, will answer from training data, sometimes flagged,
sometimes not, and you cannot tell which from the response text. Same failure as vector search
guessing at "Tatooine" in notebook 03, one layer up.

## 2. A tool is a contract

`createTool` takes four things: an `id` the model sees, a `description` it uses to decide
when to call, an `inputSchema` that validates the arguments, and an `execute` that runs.

The `description` is a prompt, not documentation. It is the entire basis for the model's
decision to call this tool, so vagueness there shows up as the tool never firing or firing
constantly.

Under the hood this reuses notebook 03's `hybridQuery`. An agent tool is a thin wrapper
over retrieval you already trust.

In [ ]:
const searchMovies = createTool({
  id: 'search_movies',
  description:
    'Search the movie collection by plot. Use this for any question about films, ' +
    'including titles, plots, and years. Returns matching movies with metadata.',
  inputSchema: z.object({
    query: z.string().describe('What to search for, as a descriptive phrase'),
    limit: z.number().optional().describe('How many results, default 5'),
  }),
  // Mastra v1 hands `execute` the validated input directly -- not wrapped in `{ context }`,
  // as older examples show. The Zod schema above has already run, so `input.query` is a
  // string by the time we get here.
  execute: async (input) => {
    console.log(`   [tool] search_movies(${JSON.stringify(input)})`);
    const results = await vector.hybridQuery({
      indexName: INDEX,
      queryVector: await embedder.embedQuery(input.query),
      query: input.query,
      paths: ['plot'],
      topK: input.limit ?? 5,
      metadataMode: 'document',
      textSearchIndexName: TEXT_INDEX,
    });
    return {
      movies: results.map((hit) => {
        const movie = hit.metadata as { title?: string; year?: number; plot?: string };
        return { title: movie.title, year: movie.year, plot: movie.plot };
      }),
    };
  },
});

console.log('Tool:', searchMovies.id);

Tool: search_movies


The `console.log` inside `execute` is there so you can see the query the *model* wrote. That
is worth watching, because it is the part you gave up control of.

## 3. The same question, with the tool attached

In [ ]:
const critic = new Agent({
  id: 'movie-critic',
  name: 'movie-critic',
  instructions: [
    'You are a film expert with access to a movie database.',
    'Always use search_movies before answering. Never answer from memory.',
    'Cite the year for each film. If the search returns nothing, say so.',
  ].join(' '),
  model: model,
  // Written out, not `{ searchMovies }`: the tslab kernel does not resolve a shorthand
  // property against a variable declared in an earlier cell.
  tools: { searchMovies: searchMovies },
});

const answered = await critic.generate(
  'Name three films in the collection about a submarine crew, with their release years.',
);

console.log('\n' + answered.text);
console.log('\nsteps:', answered.steps?.length, ' tool calls:', answered.toolCalls?.length);

   [tool] search_movies({"query":"submarine crew","limit":10})



Here are three films in the collection about a submarine crew:

1. **The Hunt for Red October** (1990) - A Soviet submarine captain violates orders and heads toward the USA in this Cold War thriller.

2. **U-571** (2000) - American submariners disguise themselves to board a German submarine and capture their Enigma cipher machine.

3. **Run Silent Run Deep** (1958) - A U.S. submarine commander becomes obsessed with sinking a particular Japanese ship, creating tension with his first officer and crew.



steps: 2  tool calls: 1


Two things to notice.

Now the films are real answers about real documents, with plots the search returned rather
than plots the model remembered.

The `[tool]` line is the part worth studying. The model wrote its own query. Here
`"submarine crew"`, a trimmed version of the question, and it also chose `limit: 5` from a
schema that only said the field was optional. It was under no obligation to pass the question
through unchanged, and on the next section's queries it does not. This is the trade you make
with tool use: retrieval quality now depends on phrasing the model invents at runtime, which
is a good argument for hybrid as the default from notebook 03: you cannot know in advance
whether a model-written query will read as keywords or as description.

And `steps` is 2, not 1. The agent made a round trip: call the tool, read the result, then
write an answer. Every tool call is another model invocation, which is where agent latency
and token cost come from.

## 4. Two tools, and letting the model choose

One tool is not much of a decision. Add a second that does something the first cannot:
filtered search, using the `year` filter field declared back in notebook 01.

In [ ]:
const searchByEra = createTool({
  id: 'search_movies_by_era',
  description:
    'Search the movie collection by plot, restricted to films released in a year range. ' +
    'Use this when the question mentions a decade, period, or "recent"/"old" films.',
  inputSchema: z.object({
    query: z.string().describe('What to search for, as a descriptive phrase'),
    fromYear: z.number().describe('Earliest release year, inclusive'),
    toYear: z.number().describe('Latest release year, inclusive'),
  }),
  execute: async (input) => {
    console.log(`   [tool] search_movies_by_era(${JSON.stringify(input)})`);
    const results = await vector.hybridQuery({
      indexName: INDEX,
      queryVector: await embedder.embedQuery(input.query),
      query: input.query,
      paths: ['plot'],
      topK: 5,
      // `year` is a declared filter field, so this is pushed into $vectorSearch
      // rather than run as an `_id: { $in: [...] }` pre-filter. Notebook 01, section 5.
      filter: { year: { $gte: input.fromYear, $lte: input.toYear } },
      metadataMode: 'document',
      textSearchIndexName: TEXT_INDEX,
    });
    return {
      movies: results.map((hit) => {
        const movie = hit.metadata as { title?: string; year?: number };
        return { title: movie.title, year: movie.year };
      }),
    };
  },
});

const curator = new Agent({
  id: 'movie-curator',
  name: 'movie-curator',
  instructions: [
    'You are a film expert with access to a movie database.',
    'Always search before answering. Pick the search tool that fits the question.',
    'Cite the year for each film.',
  ].join(' '),
  model: model,
  tools: { searchMovies: searchMovies, searchByEra: searchByEra },
});

for (const question of [
  'What heist films are in the collection?',
  'What space films from the 1970s are in the collection?',
]) {
  console.log(`\nQ: ${question}`);
  const reply = await curator.generate(question);
  console.log(reply.text.split('\n').slice(0, 6).join('\n'));
}


Q: What heist films are in the collection?


   [tool] search_movies({"query":"heist films","limit":20})


Here are the heist films in the collection:

1. **Heat** (1995) - A group of professional bank robbers start to feel the heat from police when they unknowingly leave a clue at their latest heist.

2. **The Getaway** (1972) - A recently released ex-con and his loyal wife go on the run after a heist goes awry.




Q: What space films from the 1970s are in the collection?


   [tool] search_movies_by_era({"query":"space films","fromYear":1970,"toYear":1979})


I'll search for space films from the 1970s in the collection.Based on the search results, here are the space films from the 1970s in the collection:

1. **Moonraker** (1979)
2. **Buck Rogers in the 25th Century** (1979)
3. **Message from Space** (1978)
4. **The Black Hole** (1979)


The tool descriptions did the routing. Neither question names a tool. The phrase "from the
1970s" is the whole reason the second question went to `search_movies_by_era`, and the model
translated "1970s" into `fromYear: 1970, toYear: 1979` on its own. The schema documented the
fields but never explained the decade convention.

That second call is also a filtered search running through the pushdown path from notebook 01:
`year` is a declared filter field, so Atlas applies the range inside `$vectorSearch` instead
of falling back to an `_id: { $in: [...] }` pre-filter. All four results really are from the
1970s.

Notice the model also picked `limit: 20` for the heist query, off a schema that only said the
field was optional. Values you leave to the model *will* be chosen by the model, so constrain
what matters: `z.number().max(25)` rather than a comment.

This is why `description` deserves more care than it usually gets. Tool selection is a
text-matching problem solved against your prose, so descriptions that overlap produce routing
that varies run to run, and the fix is editing English rather than code.

## 5. Memory

So far each `generate` call has been independent. Ask a follow-up and it falls apart.

In [ ]:
const forgetful = await critic.generate('Which of those is older?');
console.log(forgetful.text.split('\n').slice(0, 5).join('\n'));

I'd be happy to help you compare the ages of films, but I need you to specify which movies you're asking about. Could you please tell me which films you'd like me to compare?


There is no "those". The agent has no record of the previous exchange, so it does what it can
with a pronoun that refers to nothing.

`Memory` fixes this, and `MongoDBStore` is where it keeps the record. Two identifiers matter:

- **`thread`**: one conversation. Roughly a chat window.
- **`resource`**: who the conversation belongs to. Usually a user id. Threads are scoped to
  it, which is what stops one user's history leaking into another's.

`lastMessages: 10` is the working window: the ten most recent messages get replayed into
context on each call. Everything else stays in MongoDB, retrievable but not automatically
sent.

In [ ]:
const store = new MongoDBStore({
  id: 'cookbook-memory',
  url: cfg.mongodbUri,
  dbName: cfg.dbName,
  options: { appName: APP_NAME },
});

const memory = new Memory({
  storage: store,   // spelled out for the same tslab reason
  options: {
    lastMessages: 10,
    // Both off deliberately: semantic recall embeds every message and needs its own
    // vector index, and this cluster's three-index budget is already spent. Working
    // memory adds an extra model call per turn. Neither is needed to show persistence.
    semanticRecall: false,
    workingMemory: { enabled: false },
  },
});

const remembering = new Agent({
  id: 'movie-critic-with-memory',
  name: 'movie-critic-with-memory',
  instructions: [
    'You are a film expert with access to a movie database.',
    'Always use search_movies before answering a new question about films.',
    'For follow-up questions about films already discussed, use the conversation history.',
    'Cite the year for each film.',
  ].join(' '),
  model: model,
  tools: { searchMovies: searchMovies },
  memory: memory,
});

const THREAD = 'cookbook-nb04-thread';
const RESOURCE = 'cookbook-reader';

// Start clean, so a re-run of this notebook does not accumulate turns from the last one.
await memory.deleteThread(THREAD).catch(() => {});

const first = await remembering.generate(
  'Name three films in the collection about a submarine crew, with their release years.',
  { memory: { thread: THREAD, resource: RESOURCE } },
);
console.log('\n' + first.text);

   [tool] search_movies({"query":"submarine crew","limit":5})



Based on the search results, here are three films in the collection about a submarine crew:

1. **72 Meters** (2004) - A Russian submarine crew is stranded 72 meters below the ocean's surface.

2. **U-571** (2000) - A German submarine is boarded by disguised American submariners attempting to capture their Enigma cipher machine.

3. **The Rift** (1990) - An experimental submarine crew is sent to find out what happened to a mysteriously disappeared submarine in a submarine rift.


In [ ]:
const followUp = await remembering.generate('Which of those is older?', {
  memory: { thread: THREAD, resource: RESOURCE },
});
console.log(followUp.text);
console.log('\ntool calls on the follow-up:', followUp.toolCalls?.length ?? 0);

**The Rift** (1990) is the oldest of the three films I mentioned.



tool calls on the follow-up: 0


The pronoun resolves, and *"the three films I mentioned"* is the tell: the agent is
referring to its own previous turn.

The tool-call count on that turn is 0. It answered from replayed conversation history rather
than searching again, because the films and their years were already in context and the
instructions said to prefer history for follow-ups. That is a real cost difference: a
follow-up answered from memory is one model call, while one that re-searches is two model
calls plus an embedding request.

It is also a correctness trade. Memory is a cache of what was said, not a live read: if the
underlying data changed between turns, the follow-up answers from the stale copy. Which
behaviour you want is a decision your instructions make, so make it deliberately.

## 6. Is it actually persisted?

The agent above holds a live `Memory` object, so recall could just as easily be in-process
state. The test is a **new client**, constructed from nothing but the connection string, as
if the process had restarted.

In [ ]:
// A completely separate store and Memory -- no shared state with the agent above.
const freshStore = new MongoDBStore({
  id: 'fresh-client',
  url: cfg.mongodbUri,
  dbName: cfg.dbName,
  options: { appName: APP_NAME },
});
const freshMemory = new Memory({
  storage: freshStore,
  options: { lastMessages: 10, semanticRecall: false, workingMemory: { enabled: false } },
});

const recalled = await freshMemory.recall({
  threadId: THREAD,
  resourceId: RESOURCE,
  // `perPage` is the page size -- there is no `selectBy` on this API.
  perPage: 10,
});

console.log(`Recalled ${recalled.messages.length} messages from a client that never saw them:\n`);
for (const message of recalled.messages) {
  // Stored content is a JSON envelope: { format, parts: [...], content }.
  const raw = typeof message.content === 'string' ? message.content : JSON.stringify(message.content);
  let summary = raw;
  try {
    const parsed = JSON.parse(raw) as { parts?: Array<{ type?: string; text?: string; toolInvocation?: { toolName?: string } }> };
    summary = (parsed.parts ?? [])
      .map((part) => (part.type === 'text' ? part.text : `<${part.toolInvocation?.toolName ?? part.type}>`))
      .join(' ');
  } catch {
    // Not JSON -- print it as-is.
  }
  console.log(`  ${String(message.role).padEnd(9)} ${String(summary).slice(0, 88)}`);
}

Recalled 4 messages from a client that never saw them:



  user      Name three films in the collection about a submarine crew, with their release years.


  assistant <searchMovies> <step-start> Based on the search results, here are three films in the col


  user      Which of those is older?


  assistant **The Rift** (1990) is the oldest of the three films I mentioned.


Every turn survived, including the tool invocation: the `<searchMovies>` entry is the call
and its results, stored alongside the text. That is what makes a replayed conversation
coherent: the model sees not just what was said but what was retrieved.

`listThreads` is the other half of the API, and the one a UI needs to render a sidebar.

In [ ]:
// `resourceId` goes inside `filter`, not at the top level.
const listed = await freshMemory.listThreads({ filter: { resourceId: RESOURCE } });
for (const thread of listed.threads) {
  console.log(`${thread.id}  created ${new Date(thread.createdAt).toISOString().slice(0, 19)}  updated ${new Date(thread.updatedAt).toISOString().slice(0, 19)}`);
}
console.log('total threads for this resource:', listed.total);

cookbook-nb04-thread  created 2026-07-29T16:35:56  updated 2026-07-29T16:35:58


total threads for this resource: 1


### What the store put in your database

One thing to know before pointing `MongoDBStore` at a database you care about: it provisions
collections for its whole feature surface on connect, not just the ones you use.

In [ ]:
import { connect } from './src/utils.js';

const { client, db: getDb } = await connect(cfg.mongodbUri, { appName: APP_NAME });
const db = getDb(cfg.dbName);

const mastraCollections = (await db.listCollections().toArray())
  .map((info) => info.name)
  .filter((name) => name.startsWith('mastra_'))
  .sort();

const populated: string[] = [];
for (const name of mastraCollections) {
  const count = await db.collection(name).countDocuments();
  if (count > 0) populated.push(`${name} (${count})`);
}

console.log(`${mastraCollections.length} mastra_* collections created`);
console.log('holding data:', populated.join(', '));

31 mastra_* collections created


holding data: mastra_messages (4), mastra_threads (1)


Around thirty collections, and two of them have anything in them. The rest are for workflows,
scorers, datasets, skills, observability, and the other subsystems `MongoDBStore` backs.

They are empty and cheap, but they are also *there*, in a database that in this notebook is
MongoDB's own `sample_mflix`. In a real deployment give conversation state its own database
and keep it away from your operational collections, the same separation argument notebook 02
made about `allowWrites`, for the same reason: a store that manages its own schema should not
be sharing a namespace with data whose schema you own.

Here is the message document as stored, which is worth seeing once if you ever need to query
this data directly.

In [ ]:
const sample = await db.collection('mastra_messages').findOne({ thread_id: THREAD });
console.log('field names:', Object.keys(sample ?? {}).join(', '));
console.log('\nrole:', sample?.role);
// Note the field is `thread_id`, not `threadId` -- the API camel-cases, the documents do not.
console.log('thread_id:', sample?.thread_id);
console.log('content:', String(sample?.content).slice(0, 200));

field names: _id, id, content, createdAt, resourceId, role, thread_id, type



role: user


thread_id: cookbook-nb04-thread


content: {"format":2,"parts":[{"type":"text","text":"Name three films in the collection about a submarine crew, with their release years.","createdAt":1785342948470}],"content":"Name three films in the collect


## Housekeeping

**Created by this notebook:** the ~30 `mastra_*` collections `MongoDBStore` provisions on
first use, one conversation thread and its messages inside them, and -- if they were not
already there -- `movies_vector_index`, `movies_fulltext_index`, and the `mflix_plots`
registration.

**Deleted by this notebook:** the one thread it created, by `thread_id`, and nothing else.
`deleteThread` is scoped to that identifier, so it cannot reach a thread this run did not
create. We call it so re-running does not accumulate threads in a shared database. In an
application this is the user-facing "delete this chat" button, not routine housekeeping.

**Left in place:** the `mastra_*` collections themselves, both Atlas indexes, and the
registration. Notebooks 01 and 03 share the indexes.

**Nothing here touches `sample_mflix`'s own data.** The `embedded_movies` documents are
read, never written.

**To remove what this notebook added**, drop the `mastra_*` collections from the Atlas UI.
They are Mastra's, not yours, so this is safe -- which is also the argument in section 5
for pointing `MongoDBStore` at its own database rather than at one you care about.

In [ ]:
await freshMemory.deleteThread(THREAD);
const remaining = await freshMemory.listThreads({ filter: { resourceId: RESOURCE } });
console.log('threads remaining for this resource:', remaining.total);
console.log('messages remaining:', await db.collection('mastra_messages').countDocuments({ thread_id: THREAD }));

await client.close();
await vector.disconnect();
console.log('Disconnected. Atlas indexes untouched.');

threads remaining for this resource: 0


messages remaining: 0


Disconnected. Atlas indexes untouched.


## Recap

Retrieval became a tool, the model decided when to use it, and the conversation outlived the
client that created it.

Worth carrying forward:

- **An untooled agent will answer anyway.** It substitutes training data for your data and
  the response looks identical either way. Instructions alone do not give an agent access.
- **`description` is the routing logic.** The model picks tools by matching your prose. Two
  overlapping descriptions produce unpredictable selection.
- **`execute` receives validated input directly** in Mastra v1: `input.query`, not
  `{ context }`. Zod has already run.
- **Every tool call is another model round trip.** `steps: 2` for one search. Budget
  accordingly.
- **`thread` and `resource` are the two keys.** Threads scope a conversation; resources scope
  ownership.
- **Persistence is real and verifiable.** A store built from just the connection string
  recalls threads it never wrote, tool invocations included.
- **`MongoDBStore` provisions ~30 collections.** Give it its own database.

## Where to go next

The four notebooks together cover retrieval (`01`), bring-your-own operational collections
(`02`), hybrid search and reranking (`03`), and agents with persistent memory (`04`). The
pieces we deliberately left off (semantic recall, working memory, and Mastra workflows)
all need vector-index budget or model calls this free-tier cluster cannot spare, and each is
a natural next step on a paid cluster.

- [Mastra documentation](https://mastra.ai/docs)
- [`@mastra/mongodb` reference](https://mastra.ai/reference/storage/mongodb)
- [Atlas Vector Search](https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-overview/)
- [`$rankFusion`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/rankFusion/)